In [1]:
!pip install saliency lime shap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 6.0 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("/kaggle/temp", exist_ok=True)
!cp /kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/test.h5 /kaggle/temp/test.h5
!ls -lh /kaggle/temp/test.h5

-rw-r--r-- 1 root root 1.8G Aug 12 23:27 /kaggle/temp/test.h5



6 XAI methods together, side-by-side heatmap:

  1. Grad-CAM             — Hand-written (last conv layer)
  2. LIME                 — superpixel boundary
  3. SHAP                 — GradientExplainer, pixel-level
  4. Integrated Gradients — Google saliency library
  5. Blur Integrated Grad.— Google saliency library
  6. XRAI                 — Google saliency library (IG + segmentation)

Output: 2x4 grid per sample
    [ Original | Grad-CAM | LIME | SHAP ]
    [ IG       | Blur-IG  | XRAI | (blank) ]

Installation (Keep Internet On once in Kaggle):
    !pip install saliency lime shap -q

Execution:
    python 09_xai_six.py \
        --h5 /kaggle/temp/test.h5 \
        --ckpt /kaggle/input/.../best_cnn.pt --model cnn \
        --out_dir /kaggle/working/xai6_cnn --n 6

    # Only wrong predictions:
    python 09_xai_six.py ... --only_wrong

In [3]:
%%writefile /kaggle/working/09_xai_six.py

import argparse
import io
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from PIL import Image
from torchvision import models

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]


# ==================================================== MODEL (Exact match with training)

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        return F.max_pool2d(F.relu(self.bn(self.conv(x))), 2)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


def build_resnet(num_classes=4):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    return m


def load_model(model_type, ckpt_path, device):
    model = (SimpleCNN() if model_type == "cnn" else build_resnet()).to(device)
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])
    model.eval()
    target_layer = model.features[-1].conv if model_type == "cnn" else model.layer4[-1]
    return model, target_layer


# ==================================================== normalize

def get_mean_std(model_type):
    if model_type == "resnet":
        return IMAGENET_MEAN, IMAGENET_STD
    return np.array([0.5, 0.5, 0.5], np.float32), np.array([0.5, 0.5, 0.5], np.float32)


def normalize_for(model_type, arr01):
    mean, std = get_mean_std(model_type)
    arr = (arr01 - mean) / std
    return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1))).float()


def norm01(x):
    """Brings any map to 0-1 range."""
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# ==================================================== 1. Grad-CAM (Hand-written)

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.act = None
        self.grad = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)

    def _fwd(self, m, i, o):
        self.act = o.detach()

    def _bwd(self, m, gi, go):
        self.grad = go[0].detach()

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        logits[0, class_idx].backward()
        # Equation (4): Spatial average of gradients = weights
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        # Equation (5): Weighted combination + ReLU
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return norm01(cam[0, 0].cpu().numpy())


# ==================================================== 2. LIME

def lime_explain(model, model_type, img01, device, pred_idx):
    from lime import lime_image
    from skimage.segmentation import mark_boundaries

    def predict_fn(imgs):
        batch = torch.stack(
            [normalize_for(model_type, im.astype(np.float32)) for im in imgs]).to(device)
        with torch.no_grad():
            return torch.softmax(model(batch), dim=1).cpu().numpy()

    explainer = lime_image.LimeImageExplainer()
    exp = explainer.explain_instance(img01.astype(np.float64), predict_fn,
                                     top_labels=4, hide_color=0, num_samples=800)
    temp, mask = exp.get_image_and_mask(pred_idx, positive_only=True,
                                        num_features=6, hide_rest=False)
    return mark_boundaries(temp, mask)


# ==================================================== 3. SHAP

def shap_explain(model, x_tensor, background):
    import shap
    explainer = shap.GradientExplainer(model, background)
    sv = explainer.shap_values(x_tensor)
    arr = sv[0][0] if isinstance(sv, list) else sv[0]
    if arr.ndim == 4:
        arr = arr[..., 0]
    heat = np.abs(arr).sum(axis=0) if arr.ndim == 3 else np.abs(arr)
    return norm01(heat)


# ==================================================== 4-6. saliency library (IG, Blur-IG, XRAI)

def make_call_model_function(model, model_type, device):
    """
    Expected interface for Google saliency library.

    Images come as (N, H, W, C) numpy, in 0-1 range. We convert to tensor,
    enable requires_grad, then normalize (normalization is differentiable,
    so gradients reach the original 0-1 input).
    """
    import saliency.core as saliency_core

    mean, std = get_mean_std(model_type)
    mean_t = torch.tensor(mean).view(1, 3, 1, 1).to(device)
    std_t = torch.tensor(std).view(1, 3, 1, 1).to(device)

    def call_model_function(images, call_model_args=None, expected_keys=None):
        x = torch.from_numpy(images.transpose(0, 3, 1, 2)).float().to(device)
        x.requires_grad_(True)
        x_norm = (x - mean_t) / std_t

        out = model(x_norm)
        out = torch.softmax(out, dim=1)
        target = call_model_args["class_idx"]
        selected = out[:, target]

        grads = torch.autograd.grad(selected, x,
                                    grad_outputs=torch.ones_like(selected))[0]
        grads = grads.permute(0, 2, 3, 1).detach().cpu().numpy()  # -> (N,H,W,C)
        return {saliency_core.base.INPUT_OUTPUT_GRADIENTS: grads}

    return call_model_function


def saliency_explain(method, img01, call_fn, pred_idx, ig_steps=25,
                     blur_steps=50, batch_size=16):
    """Runs IG / Blur-IG / XRAI and returns 2D heatmap."""
    import saliency.core as saliency_core

    args = {"class_idx": pred_idx}
    baseline = np.zeros_like(img01)      # Black image = reference

    if method == "IG":
        obj = saliency_core.IntegratedGradients()
        attr = obj.GetMask(img01, call_fn, args, x_steps=ig_steps,
                           x_baseline=baseline, batch_size=batch_size)
        heat = np.abs(attr).sum(axis=2)

    elif method == "BlurIG":
        obj = saliency_core.BlurIG()
        attr = obj.GetMask(img01, call_fn, args, steps=blur_steps,
                           batch_size=batch_size)
        heat = np.abs(attr).sum(axis=2)

    elif method == "XRAI":
        obj = saliency_core.XRAI()
        params = saliency_core.XRAIParameters()
        params.algorithm = "fast"        # Fast version, otherwise takes too long
        heat = obj.GetMask(img01, call_fn, args,
                           extra_parameters=params, batch_size=batch_size)
    else:
        raise ValueError(method)

    return norm01(heat)


# ==================================================== Data helpers

def load_labels(h5_path):
    with h5py.File(h5_path, "r") as f:
        return f["test"]["label"].shape[0], f["test"]["label"][:].astype(int)


def get_patch(h5_path, idx):
    with h5py.File(h5_path, "r") as f:
        png = f["test"]["png"][idx].tobytes()
    return np.asarray(Image.open(io.BytesIO(png)).convert("RGB"),
                      dtype=np.float32) / 255.0


# ==================================================== main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--h5", default="/kaggle/temp/test.h5")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--model", choices=["cnn", "resnet"], required=True)
    ap.add_argument("--out_dir", default="/kaggle/working/xai6")
    ap.add_argument("--n", type=int, default=6)
    ap.add_argument("--only_wrong", action="store_true")
    ap.add_argument("--damage_class", default="all",
                    choices=["all", "no-damage", "minor-damage",
                             "major-damage", "destroyed"],
                    help="Show samples of this class only (based on true label)")
    ap.add_argument("--ig_steps", type=int, default=25)
    ap.add_argument("--blur_steps", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Device: {device} | model: {args.model}")

    model, target_layer = load_model(args.model, args.ckpt, device)
    gradcam = GradCAM(model, target_layer)
    call_fn = make_call_model_function(model, args.model, device)

    n_total, labels = load_labels(args.h5)
    rng = np.random.RandomState(args.seed)
    order = rng.permutation(n_total)

    print("Preparing SHAP background...")
    bg_idx = rng.choice(n_total, 20, replace=False)
    background = torch.stack(
        [normalize_for(args.model, get_patch(args.h5, i)) for i in bg_idx]).to(device)

    panel_order = ["Original", "Grad-CAM", "LIME", "SHAP",
                   "Integrated Gradients", "Blur-IG", "XRAI"]

    made = 0
    for idx in order:
        if made >= args.n:
            break

        true_idx = int(labels[idx])

        # Class filter — drop before loading image, making it much faster
        if args.damage_class != "all" and CLASSES[true_idx] != args.damage_class:
            continue

        img01 = get_patch(args.h5, idx)
        x = normalize_for(args.model, img01).unsqueeze(0).to(device)

        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        pred_idx = int(probs.argmax())

        if args.only_wrong and pred_idx == true_idx:
            continue

        panels = {"Original": ("rgb", img01)}
        blank = np.zeros(img01.shape[:2], np.float32)

        # 1. Grad-CAM
        try:
            panels["Grad-CAM"] = ("cam", gradcam(x.clone(), pred_idx))
        except Exception as e:
            print(f"  Grad-CAM fail: {e}"); panels["Grad-CAM"] = ("cam", blank)

        # 2. LIME
        try:
            panels["LIME"] = ("rgb", lime_explain(model, args.model, img01,
                                                  device, pred_idx))
        except Exception as e:
            print(f"  LIME fail: {e}"); panels["LIME"] = ("rgb", img01)

        # 3. SHAP
        try:
            panels["SHAP"] = ("cam", shap_explain(model, x, background))
        except Exception as e:
            print(f"  SHAP fail: {e}"); panels["SHAP"] = ("cam", blank)

        # 4-6. IG, Blur-IG, XRAI
        for label, meth in [("Integrated Gradients", "IG"),
                            ("Blur-IG", "BlurIG"),
                            ("XRAI", "XRAI")]:
            try:
                heat = saliency_explain(meth, img01, call_fn, pred_idx,
                                        ig_steps=args.ig_steps,
                                        blur_steps=args.blur_steps)
                panels[label] = ("cam", heat)
            except Exception as e:
                print(f"  {label} fail: {e}")
                panels[label] = ("cam", blank)

       # ---------- Plotting: 2 rows × 4 columns ----------
        mark = "OK" if pred_idx == true_idx else "WRONG"
        accent = "#2E7D32" if mark == "OK" else "#C62828"   # Green / Red

        fig, axes = plt.subplots(2, 4, figsize=(19, 9.5))
        axes = np.asarray(axes).ravel()

        for i, name in enumerate(panel_order):
            ax = axes[i]
            kind, data = panels[name]
            if kind == "rgb":
                ax.imshow(data, interpolation="nearest")
            else:
                ax.imshow(img01, interpolation="nearest")
                ax.imshow(data, cmap="jet", alpha=0.5, interpolation="bilinear")
            ax.set_title(name, fontsize=13, fontweight="bold",
                         pad=14, color="#222222")
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_edgecolor("#BBBBBB")
                s.set_linewidth(1.0)

        # Hide the unused 8th cell (bar chart panel is commented out)
        for j in range(len(panel_order), len(axes)):
            axes[j].axis("off")

        # ---------- Title ----------
        fig.suptitle(
            f"True: {CLASSES[true_idx]}     |     "
            f"Pred: {CLASSES[pred_idx]}  ({probs[pred_idx]:.2f})     |     {mark}",
            fontsize=17, fontweight="bold", color=accent, y=0.975)
        fig.text(0.5, 0.925, f"{args.model.upper()}  ·  blue label = true class",
                 ha="center", fontsize=10.5, color="#666666", style="italic")

        fig.subplots_adjust(left=0.02, right=0.98, top=0.86, bottom=0.05,
                            wspace=0.12, hspace=0.30)
        # ... suptitle / fig.text unchanged ...

        fig.subplots_adjust(left=0.02, right=0.98, top=0.8, bottom=0.05,
                            wspace=0.12, hspace=0.30)
        
        # # ---------- 8th cell: prediction probability bar chart ----------
        # axp = axes[7]
        # short = ["no-dmg", "minor", "major", "destr"]
        # colors = ["#9E9E9E"] * 4
        # colors[pred_idx] = accent
        # bars = axp.barh(range(4), probs, color=colors, height=0.6)
        # axp.set_yticks(range(4))
        # axp.set_yticklabels(short, fontsize=11)
        # axp.invert_yaxis()
        # axp.set_xlim(0, 1)
        # axp.set_xlabel("probability", fontsize=10)
        # axp.set_title("Model confidence", fontsize=13, fontweight="bold", pad=14)
        # axp.grid(axis="x", alpha=0.3, linestyle=":")
        # axp.set_axisbelow(True)
        # for b, p in zip(bars, probs):
        #     axp.text(min(p + 0.03, 0.92), b.get_y() + b.get_height()/2,
        #              f"{p:.2f}", va="center", fontsize=10)
        # # Mark the actual class label in blue-bold (arrows might overlap the text)
        # for j, tick in enumerate(axp.get_yticklabels()):
        #     if j == true_idx:
        #         tick.set_color("#1565C0")
        #         tick.set_fontweight("bold")
        # for s in ["top", "right"]:
        #     axp.spines[s].set_visible(False)

        fname = out_dir / f"{args.model}_{made:02d}_{mark}_{CLASSES[true_idx]}.png"
        fig.savefig(fname, dpi=130, facecolor="white")
        plt.close(fig)

        made += 1
        print(f"  [{made}/{args.n}] saved: {fname.name}")

    print(f"\n{made} XAI images saved -> {out_dir}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/09_xai_six.py


CNN

In [4]:
for cls in ["no-damage", "minor-damage", "major-damage", "destroyed"]:
    !python /kaggle/working/09_xai_six.py \
        --h5 /kaggle/temp/test.h5 \
        --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt \
        --model cnn --damage_class {cls} \
        --out_dir /kaggle/working/xai6_CNN_{cls} --n 20

Device: cuda | model: cnn
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 649.73it/s]
  [1/20] saved: cnn_00_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 734.72it/s]
  [2/20] saved: cnn_01_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 794.98it/s]
  [3/20] saved: cnn_02_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:00<00:00, 812.18it/s]
  [4/20] saved: cnn_03_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 764.67it/s]
  [5/20] saved: cnn_04_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 770.14it/s]
  [6/20] saved: cnn_05_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 778.70it/s]
  [7/20] saved: cnn_06_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 757.03it/s]
  [8/20] saved

Resnet

In [5]:
for cls in ["no-damage", "minor-damage", "major-damage", "destroyed"]:
    !python /kaggle/working/09_xai_six.py \
        --h5 /kaggle/temp/test.h5 \
        --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt \
        --model resnet --damage_class {cls} \
        --out_dir /kaggle/working/xai6_Resnet_{cls} --n 20

Device: cuda | model: resnet
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 446.46it/s]
  [1/20] saved: resnet_00_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 495.93it/s]
  [2/20] saved: resnet_01_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 495.30it/s]
  [3/20] saved: resnet_02_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 493.66it/s]
  [4/20] saved: resnet_03_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 495.24it/s]
  [5/20] saved: resnet_04_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 496.15it/s]
  [6/20] saved: resnet_05_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 496.49it/s]
  [7/20] saved: resnet_06_OK_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 49

CNN _ wrong

In [6]:
for cls in ["no-damage", "minor-damage", "major-damage", "destroyed"]:
    !python /kaggle/working/09_xai_six.py \
        --h5 /kaggle/temp/test.h5 \
        --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt \
        --model cnn --damage_class {cls} --only_wrong \
        --out_dir /kaggle/working/xai6_CNN_only_wrong{cls} --n 20

Device: cuda | model: cnn
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 756.89it/s]
  [1/20] saved: cnn_00_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 775.29it/s]
  [2/20] saved: cnn_01_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 757.15it/s]
  [3/20] saved: cnn_02_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 781.05it/s]
  [4/20] saved: cnn_03_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 755.87it/s]
  [5/20] saved: cnn_04_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 717.66it/s]
  [6/20] saved: cnn_05_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 794.51it/s]
  [7/20] saved: cnn_06_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 780.44it

Resnet - wrong

In [7]:
for cls in ["no-damage", "minor-damage", "major-damage", "destroyed"]:
    !python /kaggle/working/09_xai_six.py \
        --h5 /kaggle/temp/test.h5 \
        --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt \
        --model resnet --damage_class {cls} --only_wrong \
        --out_dir /kaggle/working/xai6_Resnet_only_wrong{cls} --n 20

Device: cuda | model: resnet
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 454.66it/s]
  [1/20] saved: resnet_00_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 491.56it/s]
  [2/20] saved: resnet_01_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 490.52it/s]
  [3/20] saved: resnet_02_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 487.77it/s]
  [4/20] saved: resnet_03_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 491.07it/s]
  [5/20] saved: resnet_04_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 494.31it/s]
  [6/20] saved: resnet_05_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 494.04it/s]
  [7/20] saved: resnet_06_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/80